In [ ]:
# ============================================================
# CELL 1 --NOTEBOOK 7 — OPTUNA HYPERPARAMETER OPTIMIZATION
# ============================================================

import pandas as pd
import numpy as np
import optuna
import xgboost as xgb
import joblib

from pathlib import Path

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("NOTEBOOK 7 — OPTUNA HYPERPARAMETER OPTIMIZATION")
print("=" * 70)

print(f"XGBoost version: {xgb.__version__}")
print(f"Optuna version:  {optuna.__version__}")

# Project directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "reports"
# Create output directories
for folder in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


NOTEBOOK 7 — OPTUNA HYPERPARAMETER OPTIMIZATION
XGBoost version: 3.3.0
Optuna version:  4.9.0


In [6]:
# ============================================================
# CELL 3 -LOAD SHAP-PRUNED BINARY DATA
# ============================================================

train_binary_shap = pd.read_csv(
    PROCESSED_DIR / "train_binary_shap.csv"
)

test_binary_shap = pd.read_csv(
    PROCESSED_DIR / "test_binary_shap.csv"
)

TARGET = "RISK_BINARY"

X_train = train_binary_shap.drop(
    columns=[TARGET]
)

y_train = train_binary_shap[TARGET].copy()

X_test = test_binary_shap.drop(
    columns=[TARGET]
)

y_test = test_binary_shap[TARGET].copy()

print("=" * 70)
print("SHAP-PRUNED DATA LOADED")
print("=" * 70)

print(f"X_train: {X_train.shape}")
print(f"y_train: {y_train.shape}")
print(f"X_test:  {X_test.shape}")
print(f"y_test:  {y_test.shape}")

SHAP-PRUNED DATA LOADED
X_train: (468, 49)
y_train: (468,)
X_test:  (117, 49)
y_test:  (117,)


In [7]:
#Cell 4 — Validate training/test separation
# ============================================================
# DATA VALIDATION
# ============================================================

print("=" * 70)
print("OPTUNA DATA VALIDATION")
print("=" * 70)

assert X_train.shape == (468, 49)
assert X_test.shape == (117, 49)

assert y_train.shape == (468,)
assert y_test.shape == (117,)

assert list(X_train.columns) == list(X_test.columns)

assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0

assert X_train.select_dtypes(
    exclude=np.number
).shape[1] == 0

assert X_test.select_dtypes(
    exclude=np.number
).shape[1] == 0

assert sorted(y_train.unique()) == [0, 1]
assert sorted(y_test.unique()) == [0, 1]

print("✓ Training: 468 × 49")
print("✓ Testing: 117 × 49")
print("✓ No missing predictors")
print("✓ All predictors numeric")
print("✓ Binary target validated")
print("✓ Train/test feature alignment confirmed")

print("\n✓ OPTUNA DATA VALIDATION PASSED")

OPTUNA DATA VALIDATION
✓ Training: 468 × 49
✓ Testing: 117 × 49
✓ No missing predictors
✓ All predictors numeric
✓ Binary target validated
✓ Train/test feature alignment confirmed

✓ OPTUNA DATA VALIDATION PASSED


In [8]:
#Cell 5 — Lock reproducible CV configuration
#We will use 5-fold Stratified CV.
# ============================================================
# CROSS-VALIDATION CONFIGURATION
# ============================================================

N_SPLITS = 5
CV_RANDOM_STATE = 42

cv = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=CV_RANDOM_STATE
)

print("=" * 70)
print("CROSS-VALIDATION CONFIGURATION")
print("=" * 70)

print(f"CV method:       StratifiedKFold")
print(f"Number of folds: {N_SPLITS}")
print(f"Shuffle:         True")
print(f"Random state:    {CV_RANDOM_STATE}")

print("\n✓ Test set excluded from hyperparameter optimization")

CROSS-VALIDATION CONFIGURATION
CV method:       StratifiedKFold
Number of folds: 5
Shuffle:         True
Random state:    42

✓ Test set excluded from hyperparameter optimization


In [9]:
#Cell 6 — Define Optuna search space
# ============================================================
# OPTUNA SEARCH SPACE
# ============================================================

def suggest_xgb_params(trial):

    return {
        "objective": "binary:logistic",
        "eval_metric": "logloss",

        "n_estimators": trial.suggest_int(
            "n_estimators",
            100,
            500,
            step=50
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            2,
            8
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            0.01,
            0.30,
            log=True
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            1,
            10
        ),

        "subsample": trial.suggest_float(
            "subsample",
            0.60,
            1.00
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            0.60,
            1.00
        ),

        "gamma": trial.suggest_float(
            "gamma",
            0.0,
            5.0
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            1e-8,
            10.0,
            log=True
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            1e-3,
            10.0,
            log=True
        ),

        "random_state": 42,
        "n_jobs": -1
    }

In [10]:
#Cell 7 — Define Optuna objective
#This is the core of Notebook 7..The objective is:
#max mean 5-fold ROC-AUC, using training data only.
# ============================================================
# OPTUNA OBJECTIVE
# ============================================================

def objective(trial):

    params = suggest_xgb_params(trial)

    fold_scores = []

    for fold, (train_idx, valid_idx) in enumerate(
        cv.split(X_train, y_train),
        start=1
    ):

        X_fold_train = X_train.iloc[train_idx]
        X_fold_valid = X_train.iloc[valid_idx]

        y_fold_train = y_train.iloc[train_idx]
        y_fold_valid = y_train.iloc[valid_idx]

        model = xgb.XGBClassifier(
            **params
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            verbose=False
        )

        valid_prob = model.predict_proba(
            X_fold_valid
        )[:, 1]

        fold_auc = roc_auc_score(
            y_fold_valid,
            valid_prob
        )

        fold_scores.append(fold_auc)

    mean_auc = np.mean(fold_scores)

    return mean_auc

In [11]:
#Cell 8 — Configure Optuna study
#50 trials for this dataset.That gives you a meaningful tuning budget 
# without making the notebook unnecessarily expensive
# ============================================================
# OPTUNA STUDY CONFIGURATION
# ============================================================

N_TRIALS = 50
OPTUNA_SEED = 42

sampler = optuna.samplers.TPESampler(
    seed=OPTUNA_SEED
)

study = optuna.create_study(
    direction="maximize",
    sampler=sampler,
    study_name="SP_XGBoost_ROC_AUC"
)

print("=" * 70)
print("OPTUNA STUDY CONFIGURATION")
print("=" * 70)

print(f"Optimization metric: ROC-AUC")
print(f"Direction:           maximize")
print(f"Trials:              {N_TRIALS}")
print(f"CV folds:            {N_SPLITS}")
print(f"Optuna seed:         {OPTUNA_SEED}")

print("\n✓ Study configured")

[I 2026-08-21 04:53:50,043] A new study created in memory with name: SP_XGBoost_ROC_AUC


OPTUNA STUDY CONFIGURATION
Optimization metric: ROC-AUC
Direction:           maximize
Trials:              50
CV folds:            5
Optuna seed:         42

✓ Study configured


In [12]:
#Cell 9 — Run Optuna --This is the computational cell.
# ============================================================
# RUN OPTUNA OPTIMIZATION ==(50 trials × 5 folds = up to 250 XGBoost fits.)
# ============================================================

print("=" * 70)
print("STARTING OPTUNA OPTIMIZATION")
print("=" * 70)

study.optimize(
    objective,
    n_trials=N_TRIALS,
    show_progress_bar=True
)

print("\n" + "=" * 70)
print("OPTUNA OPTIMIZATION COMPLETE")
print("=" * 70)

print(f"Completed trials: {len(study.trials)}")
print(f"Best CV ROC-AUC:  {study.best_value:.4f}")

STARTING OPTUNA OPTIMIZATION


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-08-21 04:54:50,279] Trial 0 finished with value: 0.681490070775785 and parameters: {'n_estimators': 250, 'max_depth': 8, 'learning_rate': 0.1205712628744377, 'min_child_weight': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'gamma': 0.2904180608409973, 'reg_alpha': 0.6245760287469893, 'reg_lambda': 0.2537815508265665}. Best is trial 0 with value: 0.681490070775785.
[I 2026-08-21 04:54:51,905] Trial 1 finished with value: 0.6445535284820999 and parameters: {'n_estimators': 400, 'max_depth': 2, 'learning_rate': 0.2708160864249968, 'min_child_weight': 9, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'gamma': 0.9170225492671691, 'reg_alpha': 5.472429642032198e-06, 'reg_lambda': 0.12561043700013558}. Best is trial 0 with value: 0.681490070775785.
[I 2026-08-21 04:54:52,930] Trial 2 finished with value: 0.7253937332508761 and parameters: {'n_estimators': 250, 'max_depth': 4, 'learning_rate': 0.08012737503998542, 'min_child_weigh

In [13]:
#Cell 10 — Display best parameters
# ============================================================
# BEST OPTUNA PARAMETERS
# ============================================================

print("=" * 70)
print("BEST OPTUNA RESULT")
print("=" * 70)

print(f"Best CV ROC-AUC: {study.best_value:.4f}")

print("\nBest hyperparameters:")

for parameter, value in study.best_params.items():
    print(f"{parameter:20s}: {value}")

BEST OPTUNA RESULT
Best CV ROC-AUC: 0.7706

Best hyperparameters:
n_estimators        : 100
max_depth           : 3
learning_rate       : 0.04133713629478509
min_child_weight    : 1
subsample           : 0.8493145117240506
colsample_bytree    : 0.8332908615865308
gamma               : 4.101525122344799
reg_alpha           : 0.004811499595716167
reg_lambda          : 0.012132398817719065


In [14]:
#Cell 11 — Save Optuna study results
# ============================================================
# SAVE OPTUNA RESULTS
# ============================================================

trials_df = study.trials_dataframe()

trials_path = (
    RESULTS_DIR / "optuna_sp_xgboost_trials.csv"
)

trials_df.to_csv(
    trials_path,
    index=False
)

best_params_path = (
    RESULTS_DIR / "sp_xgboost_best_params.csv"
)

pd.DataFrame([
    study.best_params
]).to_csv(
    best_params_path,
    index=False
)

print("=" * 70)
print("OPTUNA RESULTS SAVED")
print("=" * 70)

print(f"Trials:       {trials_path}")
print(f"Best params:  {best_params_path}")

assert trials_path.exists()
assert best_params_path.exists()

print("\n✓ Optuna provenance saved")

OPTUNA RESULTS SAVED
Trials:       ..\results\optuna_sp_xgboost_trials.csv
Best params:  ..\results\sp_xgboost_best_params.csv

✓ Optuna provenance saved


In [15]:
#Cell 12 — Train final SP-XGBoost
#Now,we use the best parameters to train on all 468 training observations
# ============================================================
# TRAIN FINAL SP-XGBOOST
# ============================================================

best_params = study.best_params.copy()

best_params.update({
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_jobs": -1
})

sp_xgboost = xgb.XGBClassifier(
    **best_params
)

sp_xgboost.fit(
    X_train,
    y_train,
    verbose=False
)

print("=" * 70)
print("FINAL SP-XGBOOST TRAINING COMPLETE")
print("=" * 70)

print("✓ Model trained on all 468 training observations")
print("✓ 49 SHAP-selected predictors")
print("✓ Optuna-optimized hyperparameters")
print("✓ Test set remains untouched")

FINAL SP-XGBOOST TRAINING COMPLETE
✓ Model trained on all 468 training observations
✓ 49 SHAP-selected predictors
✓ Optuna-optimized hyperparameters
✓ Test set remains untouched


In [16]:
#Cell 13 — Final test predictions
# ============================================================
# FINAL SP-XGBOOST TEST PREDICTIONS
# ============================================================

y_pred_sp = sp_xgboost.predict(
    X_test
)

y_prob_sp = sp_xgboost.predict_proba(
    X_test
)[:, 1]

print("=" * 70)
print("SP-XGBOOST TEST PREDICTIONS")
print("=" * 70)

print(f"Predictions:    {len(y_pred_sp)}")
print(f"Probabilities:  {len(y_prob_sp)}")

assert len(y_pred_sp) == 117
assert len(y_prob_sp) == 117

assert np.all(
    (y_prob_sp >= 0) &
    (y_prob_sp <= 1)
)

print("✓ 117 predictions generated")
print("✓ Probability values valid")

SP-XGBOOST TEST PREDICTIONS
Predictions:    117
Probabilities:  117
✓ 117 predictions generated
✓ Probability values valid


In [17]:
#Cell 14 — Evaluate SP-XGBoost
#Use the same metrics as Notebooks 4 and 6.
# ============================================================
# SP-XGBOOST PERFORMANCE
# ============================================================

from sklearn.metrics import (
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

roc_auc_sp = roc_auc_score(
    y_test,
    y_prob_sp
)

pr_auc_sp = average_precision_score(
    y_test,
    y_prob_sp
)

accuracy_sp = accuracy_score(
    y_test,
    y_pred_sp
)

precision_sp = precision_score(
    y_test,
    y_pred_sp,
    zero_division=0
)

recall_sp = recall_score(
    y_test,
    y_pred_sp,
    zero_division=0
)

f1_sp = f1_score(
    y_test,
    y_pred_sp,
    zero_division=0
)

cm_sp = confusion_matrix(
    y_test,
    y_pred_sp
)

tn_sp, fp_sp, fn_sp, tp_sp = cm_sp.ravel()

specificity_sp = (
    tn_sp /
    (tn_sp + fp_sp)
)

print("=" * 70)
print("SP-XGBOOST PERFORMANCE")
print("=" * 70)

print(f"ROC-AUC:       {roc_auc_sp:.4f}")
print(f"PR-AUC:        {pr_auc_sp:.4f}")
print(f"F1-score:      {f1_sp:.4f}")
print(f"Precision:     {precision_sp:.4f}")
print(f"Recall:        {recall_sp:.4f}")
print(f"Specificity:   {specificity_sp:.4f}")
print(f"Accuracy:      {accuracy_sp:.4f}")

print("\nConfusion Matrix:")
print(cm_sp)

SP-XGBOOST PERFORMANCE
ROC-AUC:       0.8138
PR-AUC:        0.7751
F1-score:      0.7059
Precision:     0.7826
Recall:        0.6429
Specificity:   0.8361
Accuracy:      0.7436

Confusion Matrix:
[[51 10]
 [20 36]]


In [19]:
#Cell 15 — Final three-model comparison
#This will become one of the most important outputs for Chapter 4.
# ============================================================
# THREE-MODEL COMPARISON
# ============================================================

print("=" * 70)
print("LOADING S-XGBOOST RESULTS")
print("=" * 70)

# Load the saved S-XGBoost results from Notebook 6
s_results_path = (
    RESULTS_DIR / "s_xgboost_binary_results.csv"
)

assert s_results_path.exists(), (
    f"S-XGBoost results not found: {s_results_path}"
)

s_saved = pd.read_csv(
    s_results_path
)

print("✓ S-XGBoost results loaded")
print(s_saved.to_string(index=False))


# ------------------------------------------------------------
# Extract S-XGBoost metrics
# ------------------------------------------------------------

s_row = s_saved.iloc[0]

roc_auc_s = float(s_row["ROC_AUC"])
pr_auc_s = float(s_row["PR_AUC"])
f1_s = float(s_row["F1"])
precision_s = float(s_row["Precision"])
recall_s = float(s_row["Recall"])
specificity_s = float(s_row["Specificity"])
accuracy_s = float(s_row["Accuracy"])


# ------------------------------------------------------------
# Create three-model comparison
# ------------------------------------------------------------

model_comparison = pd.DataFrame({

    "Model": [
        "Baseline XGBoost",
        "S-XGBoost",
        "SP-XGBoost"
    ],

    "Predictors": [
        61,
        49,
        49
    ],

    "ROC_AUC": [
        0.7567,
        roc_auc_s,
        roc_auc_sp
    ],

    "PR_AUC": [
        0.7220,
        pr_auc_s,
        pr_auc_sp
    ],

    "F1": [
        0.6296,
        f1_s,
        f1_sp
    ],

    "Precision": [
        0.6538,
        precision_s,
        precision_sp
    ],

    "Recall": [
        0.6071,
        recall_s,
        recall_sp
    ],

    "Specificity": [
        0.7049,
        specificity_s,
        specificity_sp
    ],

    "Accuracy": [
        0.6581,
        accuracy_s,
        accuracy_sp
    ]
})

print("\n" + "=" * 70)
print("BASELINE vs S-XGBOOST vs SP-XGBOOST")
print("=" * 70)

print(
    model_comparison.to_string(index=False)
)

LOADING S-XGBOOST RESULTS
✓ S-XGBoost results loaded
    Model  Features  ROC_AUC   PR_AUC       F1  Precision   Recall  Specificity  Accuracy
S-XGBoost        49  0.76815 0.739462 0.647619   0.693878 0.607143     0.754098  0.683761

BASELINE vs S-XGBOOST vs SP-XGBOOST
           Model  Predictors  ROC_AUC   PR_AUC       F1  Precision   Recall  Specificity  Accuracy
Baseline XGBoost          61 0.756700 0.722000 0.629600   0.653800 0.607100     0.704900  0.658100
       S-XGBoost          49 0.768150 0.739462 0.647619   0.693878 0.607143     0.754098  0.683761
      SP-XGBoost          49 0.813817 0.775148 0.705882   0.782609 0.642857     0.836066  0.743590


In [21]:
#Cell 16 — Save final SP-XGBoost model
# ============================================================
# SAVE FINAL SP-XGBOOST MODEL
# ============================================================

sp_model_path = (
    MODELS_DIR / "sp_xgboost_binary.pkl"
)

joblib.dump(
    sp_xgboost,
    sp_model_path
)

print("=" * 70)
print("SP-XGBOOST MODEL SAVED")
print("=" * 70)

print(f"Model: {sp_model_path}")

assert sp_model_path.exists()

print("✓ Final SP-XGBoost model saved")

SP-XGBOOST MODEL SAVED
Model: ..\models\sp_xgboost_binary.pkl
✓ Final SP-XGBoost model saved


In [22]:
#Cell 17 — Save final comparison
# ============================================================
# SAVE FINAL MODEL COMPARISON
# ============================================================

comparison_path = (
    RESULTS_DIR / "three_model_comparison.csv"
)

model_comparison.to_csv(
    comparison_path,
    index=False
)

print("=" * 70)
print("MODEL COMPARISON SAVED")
print("=" * 70)

print(f"File: {comparison_path}")

assert comparison_path.exists()

print("✓ Three-model comparison saved")

MODEL COMPARISON SAVED
File: ..\results\three_model_comparison.csv
✓ Three-model comparison saved


In [23]:
# ============================================================
# CELL 18 --NOTEBOOK 7 — FINAL VALIDATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 7 — FINAL VALIDATION")
print("=" * 70)

assert X_train.shape == (468, 49)
assert X_test.shape == (117, 49)

assert len(study.trials) == N_TRIALS

assert np.isfinite(study.best_value)

assert len(y_pred_sp) == 117
assert len(y_prob_sp) == 117

assert np.all(
    (y_prob_sp >= 0) &
    (y_prob_sp <= 1)
)

assert np.isfinite(roc_auc_sp)
assert np.isfinite(pr_auc_sp)
assert np.isfinite(f1_sp)
assert np.isfinite(precision_sp)
assert np.isfinite(recall_sp)
assert np.isfinite(specificity_sp)
assert np.isfinite(accuracy_sp)

assert cm_sp.shape == (2, 2)

assert sp_model_path.exists()
assert comparison_path.exists()

print("✓ SHAP-selected predictors: 49")
print("✓ Training data:             468 × 49")
print("✓ Test data:                 117 × 49")
print(f"✓ Optuna trials:             {N_TRIALS}")
print(f"✓ CV folds:                  {N_SPLITS}")
print(f"✓ Best CV ROC-AUC:           {study.best_value:.4f}")
print("✓ Test set remained isolated during tuning")
print("✓ Final test predictions:    117")
print("✓ Probabilities valid")
print("✓ All metrics finite")
print("✓ Confusion matrix valid")
print("✓ SP-XGBoost model saved")
print("✓ Optuna provenance saved")

print("\n" + "=" * 70)
print("✓ NOTEBOOK 7 SP-XGBOOST VALIDATION PASSED")
print("=" * 70)

NOTEBOOK 7 — FINAL VALIDATION
✓ SHAP-selected predictors: 49
✓ Training data:             468 × 49
✓ Test data:                 117 × 49
✓ Optuna trials:             50
✓ CV folds:                  5
✓ Best CV ROC-AUC:           0.7706
✓ Test set remained isolated during tuning
✓ Final test predictions:    117
✓ Probabilities valid
✓ All metrics finite
✓ Confusion matrix valid
✓ SP-XGBoost model saved
✓ Optuna provenance saved

✓ NOTEBOOK 7 SP-XGBOOST VALIDATION PASSED
